# NASA battery Dataset EDA

For the pruposes of this project, I will be using the NASA battery dataset, which contains data from 4 different batteries. Each battery has been cycled until failure, and the dataset includes various measurements such as voltage, current, temperature, and capacity over time.

## Data Structure

Originally the dataset is in `.mat` format, converted to `.csv` for easier handling.
Each discharge cycle is one CSV file. Only discharge cycles are used — charge and impedance cycles are excluded.

Columns:
- `time_s`: Seconds elapsed since discharge start. Irregular spacing (~10Hz but not exact).
- `voltage_V`: Battery terminal voltage (V). Starts ~4.2V, falls to cutoff ~2.7V. Most informative signal for SOC.
- `current_A`: Output current (A). Negative during discharge (~-2A constant). Negative = energy leaving battery.
- `temperature_C`: Battery surface temperature (°C). Rises during discharge due to internal resistance heating.
- `capacity_Ah`: Total charge delivered this cycle (Ah). **Scalar repeated per row.** Decreases across cycles as the battery ages. Used as Coulomb counting denominator.
- `soc`: Ground truth State of Charge ∈ [0, 1], computed via Coulomb counting using per-cycle `capacity_Ah`.
- `cycle_index`: Original cycle number in the `.mat` file — proxy for battery age.
- `battery`: Battery identifier e.g. `B0005`.

The dataset contains 4 batteries (B0005, B0006, B0007, B0018). For initial training and 
exploration, only `B0005` is used - it is the most commonly referenced battery in the 
literature, has the most cycles (~168), and provides a clean room-temperature constant-current 
discharge profile. B0006 is reserved as a held-out test for cross-battery generalization, 
which is a natural extension of this work.

# Visualization

## Imports

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

### Load in data

In [ ]:
DATA_DIR = "data/processed/B0005/"
# we need to load all the cycle csvs and concatenate them into a single dataframe
cycle_dfs = []
for filename in sorted(os.listdir(DATA_DIR)):
    if filename.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATA_DIR, filename))
        cycle_dfs.append(df)

print(f"{len(cycle_dfs)} Discharge Cycles")
print(f"")

168 Discharge Cycles


## Capacity fade across cycles

In [ ]:
# first we need to get every cycles' capacities and number
capacities = []
cycle_numbers = []


for df in cycle_dfs:
    cycle_capacity = df["capacity_Ah"].iloc[0]
    cycle_number = df["cycle_index"].iloc[0]
    capacities.append(cycle_capacity)
    cycle_numbers.append(cycle_number)
